## What is Maize (and Maize-contrib)?

**Maize** is a *graph-based workflow manager* for computational pipelines.

- **Workflows are graphs**: you connect *nodes* with *channels*.
- **Nodes are units of work**: each node runs as its **own process**.
- **Ports define communication**:
  - **Inputs** receive data.
  - **Outputs** send data.
  - Specialized ports exist (e.g., `MultiInput` for many-to-one).
- **Execution is data-flow driven**: a node typically blocks until it can receive what it needs, then produces outputs for downstream nodes.

**Maize-contrib** is an extension package that provides *domain-specific nodes* (e.g., docking, cheminformatics utilities, format conversions). You use Maize itself to orchestrate the graph; you use Maize-contrib when you want pre-built nodes for common computational chemistry tasks.

## Goal of this notebook

- Verify Maize is installed and runnable from the command line.
- Create two tiny custom nodes to understand the core concepts:
  - `Example`: sends a string from a **Parameter** to an **Output**.
  - `Taker`: receives from a **MultiInput** and logs what it got.
- Build, visualize, and execute a minimal workflow.
- Export the same workflow to a **YAML** file so you see how Maize workflows can be serialized.

**Prerequisites:** Complete [installation.md](../installation.md) first.

**For more information:** Please, see [the documentation](https://molecularai.github.io/maize/)

In [ ]:
# to set several environment variables for the workshop
import workshop_setup

In [ ]:
# Checking the installation of Maize by running the help command to see if it executes without errors. If Maize is installed correctly, this command should display the help information for Maize without any issues.
!maize --help

In [ ]:
# Importing necessary classes and functions from the Maize library to create nodes and workflows for computational tasks.
from maize.core.interface import Parameter, Output, MultiInput, Input
from maize.core.node import Node
from maize.core.workflow import Workflow


### Build, visualize, and run a workflow

In the next cell we will:

- **Create a `Workflow`** (a named graph).
- **Add three nodes**:
  - two `Example` nodes, each configured with a different `data` **parameter**
  - one `Taker` node that will **receive** from both
- **Connect ports** so both `Example.out` outputs feed into `Taker.inp` (`MultiInput`).
- **Validate** the graph with `flow.check()`.

After that:

- `flow.visualize()` draws the graph so you can sanity-check the wiring.
- `flow.execute()` runs the graph and you should see the `Taker` node log the combined message.

In [ ]:
class Example(Node):
    """" Example node that demonstrates how to create a custom node in Maize.
    A node that sends a string from a parameter to an output.
    """
    data: Parameter[str] = Parameter()
    out: Output[str] = Output()
    def run(self) -> Node:
        self.out.send(self.data.value)
class Taker(Node):
    """A node that receives data from an Example node and prints it."""
    inp: MultiInput[str] = MultiInput()
    def run(self):
        result = " ".join([inp.receive() for inp in self.inp])
        self.logger.info(f"Received data: {result}")

    


### Add the nodes and connect them
In the cell below, we add the nodes into workflow. 

Pay attention, the example node receives an input and returns an output but Taker nodes expects MultiInput therefore we add a second example node with another input.

Then, we should connect the nodes according to their input and output

In [ ]:
flow = Workflow(name="Example Workflow")
first_node = flow.add(Example, name="Example Node", parameters={"data": "Hello My People!"}) 
second_node = flow.add(Example, name="Second Node", parameters={"data": "Hello Maize Again!"})
taker = flow.add(Taker, name="Taker Node")
flow.connect(first_node.out, taker.inp)
flow.connect(second_node.out, taker.inp)
flow.check()


In [ ]:
flow.visualize()



This line executes the workflow, running all the nodes in the correct order based on their dependencies.
 When executed, it will run the Example nodes to send their data to the Taker node, which will then print the received data.


In [ ]:
flow.execute()


### Exporting a workflow (YAML)

Maize workflows don’t have to be defined only in Python. You can also **serialize** a workflow definition to a file (e.g., YAML/TOML/JSON) and run it later.

In the next cell we will:

- Build a second tiny workflow (`Example → LogResult`).
- Visualize it.
- Write it to `example_workflow.yaml` using `flow_yaml.to_file(...)`.

Then we’ll print the file contents so you can see:

- the **nodes** (with their names and types)
- the **parameters** (e.g., the `data` string)
- the **connections** (which output is wired to which input)


In [ ]:

from maize.steps.io import LogResult

flow_yaml = Workflow(name="Example Yaml")
ex = flow_yaml.add(Example, name="Example Node", parameters={"data": "Hello from YAML!"})
log = flow_yaml.add(LogResult, name="Log Node")
flow_yaml.connect(ex.out, log.inp)
flow_yaml.check()
flow_yaml.visualize()
flow_yaml.to_file("example_workflow.yaml")



In [ ]:
!cat example_workflow.yaml